# 입찰메이트 RAG — 실험 히스토리 v10

**작성일** : 2026-05-26  
**담당** : 한의정 (Retrieval + Generation)  
**목적** : v8 이후 Generation E2E 평가 파이프라인 구축 및 실행 전 과정 기록

---

## 목차

1. [v8 이후 주요 변경 사항 요약](#1-v8-이후-주요-변경-사항-요약)
2. [시나리오 구조 개편 (A-1/A-2/B → 임베딩_LLM 12개 조합)](#2-시나리오-구조-개편)
3. [Generation 모듈 수정 이력](#3-generation-모듈-수정-이력)
4. [GCP 환경 이슈 해결](#4-gcp-환경-이슈-해결)
5. [E2E 평가 파이프라인 구조 변경](#5-e2e-평가-파이프라인-구조-변경)
6. [E2E 실행 현황 및 결과](#6-e2e-실행-현황-및-결과)
7. [파일 구성 최종 현황](#7-파일-구성-최종-현황)
8. [수정 필요 항목 (오류 목록)](#8-수정-필요-항목-오류-목록)
9. [OpenRouter / API 모델 현황](#9-openrouter--api-모델-현황)
10. [파인튜닝 계획](#10-파인튜닝-계획)
11. [다음 실행 순서](#11-다음-실행-순서)
12. [GCP 서버 환경 설정](#12-gcp-서버-환경-설정)

---

## 1. v8 이후 주요 변경 사항 요약

| 항목 | v8 상태 | v10 결과 |
|---|---|---|
| Generation E2E 평가 (C타입) | ⬜ | ✅ 완료 (KURE_GEMMA vs SMALL_OPENAI, 61개) |
| 시나리오 명명 체계 개편 | A-1/A-2/B | ✅ 임베딩_LLM 조합으로 전환 |
| Context 사전 추출 | ⬜ | ✅ extract_contexts.py 구축 및 실행 완료 |
| ABDE 통합 E2E | ⬜ | 🔄 실행 중 (nohup, GEMMA 진행 중) |
| LoRA 파인튜닝 | ⬜ | ⬜ 베이스라인 완료 후 예정 |
| Judge 모델 | gpt-5.4-mini + llama-4 | gpt-5-mini + gemma-4-26b (OR) 로 변경 |

---

## 2. 시나리오 구조 개편

### 기존 (A-1/A-2/B)

```
A-1: KURE-v1 임베딩 + Gemma 로컬
A-2: KoE5 임베딩 + LLaMA 로컬
B  : OpenAI API
```

### 변경 후 (임베딩_LLM 12개 조합)

```
임베딩 3개: KURE / KOE5 / SMALL
LLM    4개: GEMMA / QWEN / PHI / OPENAI

조합 12개:
  KURE_GEMMA, KURE_QWEN, KURE_PHI, KURE_OPENAI
  KOE5_GEMMA, KOE5_QWEN, KOE5_PHI, KOE5_OPENAI
  SMALL_GEMMA, SMALL_QWEN, SMALL_PHI, SMALL_OPENAI
```

### 코드 변경 내역

**`retrieval_interface_F.py` — EMBED_CONFIG**
```python
# 기존
EMBED_CONFIG = {'A-1': ..., 'A-2': ..., 'B': ...}
# 변경
EMBED_CONFIG = {'KURE': ..., 'KOE5': ..., 'SMALL': ...}
```

**`retrieval_interface_F.py` — _PROMPT_TEMPLATES**
```python
# 기존
{'A-1': Gemma포맷, 'A-2': LLaMA포맷, 'B': OpenAI포맷}
# 변경
{'GEMMA': ..., 'QWEN': ..., 'PHI': ..., 'OPENAI': ...}
```

**`generation_interface.py` — get_generator()**
```python
# 기존
if scenario in ('A-1', 'A-2'): return LocalHFGenerator()
elif scenario == 'B': return APIGenerator()

# 변경
llm_key = scenario.split('_')[1]  # 'KURE_GEMMA' → 'GEMMA'
if llm_key == 'GEMMA': return LocalHFGenerator(scenario='GEMMA')
elif llm_key in ('QWEN', 'PHI'): return OpenRouterGenerator(scenario=llm_key)
elif llm_key == 'OPENAI': return APIGenerator(scenario='OPENAI')
```

**`retrieval_eval_F.py` — BidMateEvaluator**
```python
# embed_key 추출 추가
embed_key = scenario.split('_')[0] if '_' in scenario else scenario
col_name = collection_name or EMBED_CONFIG[embed_key]['collection']
```

---

## 3. Generation 모듈 수정 이력

### 3-1. AutoModelForCausalLM → AutoModelForImageTextToText

**문제**: `google/gemma-3-4b-it`는 텍스트+이미지 멀티모달 모델이라 `AutoModelForCausalLM`으로 로드 시 `vision_tower` 레이어에서 변환 오류 발생.

```python
# 기존
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
self._model = AutoModelForCausalLM.from_pretrained(...)

# 수정
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig
self._model = AutoModelForImageTextToText.from_pretrained(...)
```

### 3-2. 양자화 비활성화 (_QUANTIZE=False)

**문제**: `bitsandbytes` 4-bit 양자화 시 `libnvJitLink.so.13` CUDA 라이브러리 경로 불일치로 RuntimeError 발생.

**해결**: `_QUANTIZE=False`로 변경. L4 24GB VRAM에서 bfloat16 전체 로드 (~9GB) 가능.

```python
_QUANTIZE = False  # bfloat16 로드, bitsandbytes 없이 동작
```

단, `LD_LIBRARY_PATH` 설정으로 해결 가능:
```bash
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/opt/jhub-venv/lib/python3.12/site-packages/nvidia/cu13/lib
```

### 3-3. OpenAI 모델명 및 파라미터 수정

| 항목 | 기존 | 수정 |
|---|---|---|
| 모델명 | `gpt-5.4-mini` | `gpt-5-mini` (팀 허용 모델 목록 기준) |
| 토큰 파라미터 | `max_tokens` | `max_completion_tokens` (gpt-5-mini 전용) |
| temperature | 0.1 | 1 (gpt-5-mini는 기본값만 지원) |

```python
_B_MODEL_ID = 'gpt-5-mini'
_B_USE_COMPLETION_TOKENS = True  # max_completion_tokens 사용
_API_GEN_PARAMS = {
    'max_tokens' : 512,
    'temperature': 1,    # gpt-5-mini 기본값만 지원
    'top_p'      : 1.0,
}
```

### 3-4. fastapi_app.py 버그 수정

```python
# 기존 (오류)
_bid_app.__class__.__mro__[0].__dict__.get('_MAX_HISTORY_MSGS', 20)

# 수정
from serving_main import _MAX_HISTORY_MSGS
```

### 3-5. eval_quant_judge_dual.py 수정

```python
# max_completion_tokens 자동 분기 추가
_JUDGE_1_USE_COMPLETION_TOKENS = True  # gpt-5-mini: True
token_param = {'max_completion_tokens': 20} if use_completion_tokens else {'max_tokens': 20}

# 출력 컬럼명 시나리오 독립적으로 변경
# 기존: gemma_faithfulness, gpt_faithfulness
# 변경: a_faithfulness, b_faithfulness (SCENARIO_A/B 환경변수 기반)
```

---

## 4. GCP 환경 이슈 해결

| 이슈 | 원인 | 해결 |
|---|---|---|
| `GatedRepoError` | HuggingFace 로그인 안 됨 | `hf auth login` (토큰명: bidmate) |
| `accelerate` 없음 | 패키지 미설치 | `pip install accelerate` |
| `libnvJitLink.so.13` 없음 | bitsandbytes CUDA 경로 불일치 | `export LD_LIBRARY_PATH=.../nvidia/cu13/lib` |
| Gemma 로드 오류 | `AutoModelForCausalLM` → 멀티모달 모델 부적합 | `AutoModelForImageTextToText`로 변경 |
| bitsandbytes 양자화 실패 | CUDA 13 버전 불일치 | `_QUANTIZE=False` (bfloat16 로드) |
| `temperature=0.1` 오류 | `gpt-5-mini`는 기본값(1)만 지원 | `_API_GEN_PARAMS['temperature']=1` |
| `nohup` Exit 127 | API 키 특수문자로 인한 파싱 오류 | `export` 방식으로 환경변수 설정 |

---

## 5. E2E 평가 파이프라인 구조 변경

### 5-1. Context 사전 추출 도입

**기존**: E2E 생성 시마다 `get_context()` 호출 → Retrieval 반복 실행 → 시간 낭비

**변경**: `extract_contexts.py`로 context 1회 추출 → CSV 저장 → E2E에서 재사용

```bash
BIDMATE_ENV=gcp python extract_contexts.py --embed KURE
→ eval_contexts_abde_kure.csv  (A/B/D/E 516개)
→ eval_contexts_c_kure.csv     (C타입 63개)
```

**효과**: Retrieval 시간 완전 제거, Generation만 실행 → 시간 대폭 절약

### 5-2. eval_e2e_abde.py 통합

| 기존 파일 | 변경 | 이유 |
|---|---|---|
| `eval_e2e_d_e_type_legacy.py` | → `eval_e2e_abde.py`로 통합 | GEMMA 로드 1회로 절약 |
| `eval_e2e_a_b_type_legacy.py` | → `eval_e2e_abde.py`로 통합 | 동일 이유 |

**legacy 파일**: `get_context()` 직접 호출 방식, 보관용으로 `_legacy` 접미사 유지

### 5-3. 파일명 규칙 확정

```
eval_results/generation/
├── eval_contexts_abde_{embed}.csv     ← 임베딩별 A/B/D/E context
├── eval_contexts_c_{embed}.csv        ← 임베딩별 C타입 context
├── e2e_abde_{embed}_comparison.csv    ← ABDE 최종 결과
├── e2e_ctype_{embed}_comparison.csv   ← C타입 최종 결과
├── e2e_abde_{embed}_mid_{scenario}.csv  ← 중간파일 (재시작용)
├── e2e_ctype_{embed}_mid_{scenario}.csv
├── e2e_all_comparison.csv             ← C+ABDE 합친 파일 (Judge 입력)
├── quantitative_scores.csv            ← Judge 채점 결과
├── qual_error_analysis.csv
├── qual_side_by_side.csv
└── qual_ctype_tracking.csv
```

### 5-4. EMBED 환경변수 도입

임베딩 모델별로 context 파일을 구분하기 위해 `EMBED` 환경변수 추가:

```bash
EMBED=KURE python eval_e2e_abde.py   # eval_contexts_abde_kure.csv 사용
EMBED=KOE5 python eval_e2e_abde.py   # eval_contexts_abde_koe5.csv 사용
```

---

## 6. E2E 실행 현황 및 결과

### 6-1. C타입 (완료)

| 항목 | 내용 |
|---|---|
| 대상 | C타입 61개 (히스토리 있는 것) |
| GEMMA 소요시간 | 1시간 34분 |
| OPENAI 소요시간 | 2분 14초 |
| 결과 파일 | `e2e_ctype_comparison.csv` (구버전, context 없이 생성) |

**⚠️ 주의**: 기존 C타입 결과는 context CSV 재사용 전 버전이라 새 파이프라인과 다름. 재실행 필요.

**답변 품질 샘플 비교:**
- **GEMMA**: 문서 내용 나열, 질문에 직접 답하지 않음 → 품질 낮음
- **OPENAI(gpt-5-mini)**: 핵심 답변 + 출처 명시 → 품질 높음

예시:
```
Q: 제조사(SAP社)의 기술지원 종료 시점은?
GEMMA: (문서 청크 그대로 나열...)
OPENAI: 문서에서 언급하는 제조사(SAP社)의 기술지원 종료 시점은 '27년(2027년)입니다.
        출처: 과 업 지 시 서 - 차세대 통합정보시스템(ERP) 구축 -
```

### 6-2. A/B/D/E타입 (진행 중)

| 항목 | 내용 |
|---|---|
| 대상 | A172 + B214 + D65 + E65 = 516개 |
| 현재 상태 | KURE_GEMMA 실행 중 (nohup, `abde_run.log`) |
| 속도 | 질문당 약 35초 |
| 예상 완료 | GEMMA ~5시간, OPENAI ~2분 |

**이전 실패 이력:**
1. 첫 실행: context CSV 파일명 불일치 (`eval_contexts_abde.csv` vs `eval_contexts_abde_kure.csv`) → context 빈 문자열로 생성 → 무효
2. temperature 오류: gpt-5-mini는 `temperature=0.1` 미지원 → `temperature=1`로 수정
3. 현재 실행: 파일명 수정 후 정상 실행 중

---

## 7. 파일 구성 최종 현황

```
2Team_Project/hej/
├── retrieval_interface_F.py        # Retrieval 핵심
├── retrieval_eval_F.py             # Retrieval 평가 전용
├── run_eval_F.py                   # 배치 평가 실행
├── generation_interface.py         # LLM 생성 모듈
├── serving_main.py                 # 통합 서빙 파이프라인
├── fastapi_app.py                  # FastAPI 엔드포인트
├── extract_contexts.py             # Context 사전 추출 (신규)
├── eval_e2e_abde.py                # A/B/D/E E2E 평가 (신규, context CSV 재사용)
├── eval_e2e_ctype.py               # C타입 E2E 평가 (context CSV 재사용으로 수정)
├── eval_e2e_a_b_type_legacy.py     # A/B E2E 구버전 (get_context 직접 호출, 보관용)
├── eval_e2e_d_e_type_legacy.py     # D/E E2E 구버전 (get_context 직접 호출, 보관용)
├── eval_quant_judge.py             # 단일 Judge 정량 평가
├── eval_quant_judge_dual.py        # 듀얼 Judge 정량 평가
└── eval_qual_analyzer.py           # 정성 평가 리포트
```

---

## 8. 수정 필요 항목 (오류 목록)

### 완료

| 파일 | 수정 내용 |
|---|---|
| `eval_quant_judge_dual.py` | `qwen3-235b-a22b:free` → `google/gemma-4-26b-a4b-it:free` |
| `eval_quant_judge_dual.py` | 경로 `e2e_all_comparison.csv` → `generation/e2e_all_comparison.csv` |
| `eval_quant_judge_dual.py` | 경로 `quantitative_scores.csv` → `generation/quantitative_scores.csv` |
| `eval_quant_judge.py` | `qwen3-235b-a22b:free` → `google/gemma-4-26b-a4b-it:free` |
| `eval_quant_judge.py` | 경로 `e2e_ctype_comparison.csv` → `generation/e2e_ctype_comparison.csv` |
| `eval_quant_judge.py` | 경로 `quantitative_scores.csv` → `generation/quantitative_scores.csv` |
| `eval_qual_analyzer.py` | 경로 `e2e_ctype_comparison.csv` → `generation/e2e_all_comparison.csv` |
| `eval_qual_analyzer.py` | 경로 `quantitative_scores.csv` → `generation/quantitative_scores.csv` |
| `run_eval_F.py` | `from retrieval_eval import` → `from retrieval_eval_F import` |
| `serving_main.py` | `gpt-5.4-mini` → `gpt-5-mini` |

### 미완료 (실행 전 수정 필요)

| 파일 | 오류 | 수정 방법 |
|---|---|---|
| `eval_qual_analyzer.py` | `_E2E_PATH`가 `e2e_all_comparison.csv`인데 C+ABDE 합치기 전까지 없음 | ABDE 완료 후 합치기 실행 |

---

## 9. OpenRouter / API 모델 현황

### OpenRouter 무료 모델

| 모델 ID | 상태 | 비고 |
|---|---|---|
| `qwen/qwen3-14b:free` | ❌ 404 | 없어짐 |
| `qwen/qwen3-235b-a22b:free` | ❌ 404 | 코드에서 교체 완료 |
| `qwen/qwen3.6-plus-preview:free` | ❌ 404 | 없음 |
| `google/gemma-4-26b-a4b-it:free` | ✅ 작동 확인 | 현재 Judge 2로 사용 예정 |
| `microsoft/phi-4-mini-instruct:free` | ❓ 미확인 | 테스트 필요 |

### 팀 OpenAI API

| 항목 | 내용 |
|---|---|
| 허용 모델 | `gpt-5-mini`, `gpt-5-nano`, `text-embedding-3-small` |
| 한도 | $20 (현재 $1.55 사용, $18.45 남음) |
| 주의 | `gpt-5-mini`는 `temperature` 기본값(1)만 지원 |

### API 키 관리

| 키 | 노출 횟수 | 상태 |
|---|---|---|
| 팀 OpenAI API 키 | 4회 | ⚠️ 재발급 필요 |
| OpenRouter API 키 | 3회 | ⚠️ 재발급 필요 |
| HuggingFace 토큰 (bidmate) | 0회 | ✅ 안전 |

---

## 10. 파인튜닝 계획

**방식**: LoRA (bfloat16)
- QLoRA 보류: bitsandbytes CUDA 문제
- L4 24GB에서 4B급 모델 + LoRA 오버헤드 ~12GB → 가능

**순서**: Gemma 3-4b-it → Qwen3-4B-Instruct → Phi-4-mini-instruct

**학습 데이터**: eval 579개 Q&A 재사용
- leakage 문제 있으나 시간 제약으로 감수
- 발표 시 명시 예정: "실제 서비스에서는 별도 학습 데이터 구축 필요"

```python
# 학습 데이터 포맷
{
    'instruction': row['question'],
    'input'      : f"[참고 문서]\n{row['retrieved_context']}",
    'output'     : row['ans_small_openai']  # GPT 답변을 정답으로
}
```

**선행 조건**: 베이스라인 E2E + Judge 채점 완료 후 진행

---

## 11. 다음 실행 순서

```bash
# 1. ABDE 완료 확인
tail -20 /home/euijeong/2Team_Project/hej/abde_run.log

# 2. C타입 재실행 (context CSV 재사용 버전)
export EMBED=KURE SCENARIO_A=KURE_GEMMA SCENARIO_B=SMALL_OPENAI
export BIDMATE_ENV=gcp OPENAI_API_KEY=새키
python eval_e2e_ctype.py

# 3. C+ABDE 합치기
python3 -c "
import pandas as pd
from pathlib import Path
d = Path('/home/euijeong/2Team_Project/hej/eval_results/generation')
c  = pd.read_csv(d / 'e2e_ctype_kure_comparison.csv')
ab = pd.read_csv(d / 'e2e_abde_kure_comparison.csv')
merged = pd.concat([c, ab], ignore_index=True)
merged.to_csv(d / 'e2e_all_comparison.csv', index=False, encoding='utf-8-sig')
print(f'완료: {len(merged)}개')
"

# 4. Judge 채점
SCENARIO_A=KURE_GEMMA SCENARIO_B=SMALL_OPENAI \
OPENAI_API_KEY=새키 OPENROUTER_API_KEY=새키 \
python eval_quant_judge_dual.py

# 5. 정성 평가
SCENARIO_A=KURE_GEMMA SCENARIO_B=SMALL_OPENAI SCORE_PREFIX=avg \
python eval_qual_analyzer.py
```

---

## 12. GCP 서버 환경 설정

```bash
# 매 세션 필요
source /home/euijeong/2Team_Project/serve_env/bin/activate
cd /home/euijeong/2Team_Project/hej
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/opt/jhub-venv/lib/python3.12/site-packages/nvidia/cu13/lib

# 환경변수
export EMBED=KURE
export SCENARIO_A=KURE_GEMMA
export SCENARIO_B=SMALL_OPENAI
export BIDMATE_ENV=gcp
export OPENAI_API_KEY=새키
export OPENROUTER_API_KEY=새키

# 백그라운드 실행
nohup python eval_e2e_abde.py > abde_run.log 2>&1 &

# 로그 확인
tail -f abde_run.log
```

### ChromaDB 컬렉션 목록

```
bidmate_chunks_all       bidmate_chunks_all_A-1    bidmate_chunks_all_A-2
bidmate_chunks_all_B     bidmate_kure              bidmate_kh_v3_A-1
bidmate_kh_v3_A-2        bidmate_kh_v3_B           bidmate_kh_fixed_v1_A-1
bidmate_kh_fixed_v1_A-2  bidmate_kh_fixed_v1_B     bidmate_kh_fixed_v2_A-1
bidmate_kh_fixed_v2_A-2  bidmate_kh_fixed_v2_B     bidmate_retrieval_v1
bidmate_retrieval_openai_v1  bidmate_retrieval_koe5_v1
```

**현재 사용 중:**
- ABDE: `bidmate_kure` (chunks_all + KURE 임베딩)
- C타입: `bidmate_kh_v3_A-1` (kh_v3 + KURE 임베딩, use_hybrid=True)


---
# 입찰메이트 RAG — 서빙 파이프라인 구조


```
사용자 쿼리
    │
    ▼
┌─────────────────────────────────────────────────────┐
│  serving_main.py — BidMateApp.chat()                │
│                                                     │
│  _classify_query()                                  │
│    ├─ 지시어 키워드 포함? ('그 ', '앞서 ', '해당' …)       │
│    └─ 직전 발화와 형태소 오버랩 없음?                      │
│         ↓ Yes → C타입    ↓ No → A/B/D/E타입           │
└─────────────────────────────────────────────────────┘
    │                           │
    ▼                           ▼
_retriever_ctx              _retriever_main
kh_v3.json                  chunks_all.json
use_hybrid=True             use_hybrid=False
(Child-to-Parent)
    │                           │
    └───────────┬───────────────┘
                ▼
┌─────────────────────────────────────────────────────┐
│  retrieval_interface_F.py — BidMateRetriever        │
│                                                     │
│  ① 메타데이터 Hard Filter  agency/year 퍼지 매칭         │
│  ② 쿼리 분해               기관명 앵커링 (B타입)          │
│  ③ Dense 검색 (k=15)       KURE-v1 임베딩 + ChromaDB  │
│  ④ Sparse 검색 (k=15)      BM25 + Kiwi 형태소 분석     │
│  ⑤ RRF 융합                Reciprocal Rank Fusion   │
│  ⑥ Soft Boost              has_table×1.10 / has_number×1.05 │
│  ⑦ MMR 재정렬              λ=0.6, Top-20 후보        │
│  ⑧ Reranker                bge-reranker-v2-m3        │
│                            Top-15 → Top-5 컷         │
│                ↓                                    │
│           Top-5 청크 반환                            │
└─────────────────────────────────────────────────────┘
                │
                ▼
┌─────────────────────────────────────────────────────┐
│  retrieval_interface_F.py — build_prompt()          │
│                                                     │
│  시나리오별 프롬프트 포맷 분기                         │
│    GEMMA  → <start_of_turn>user ... <start_of_turn>model │
│    QWEN   → Qwen 포맷                               │
│    PHI    → Phi 포맷                                │
│    OPENAI → {'system': ..., 'user': ...}            │
└─────────────────────────────────────────────────────┘
                │
                ▼
┌─────────────────────────────────────────────────────┐
│  generation_interface.py — generator.generate()     │
│                                                     │
│  GEMMA  → LocalHFGenerator                         │
│             google/gemma-3-4b-it                    │
│             bfloat16, _QUANTIZE=False               │
│             AutoModelForImageTextToText             │
│                                                     │
│  QWEN   → OpenRouterGenerator                      │
│  PHI    → OpenRouterGenerator                      │
│             (OpenRouter 무료 모델)                   │
│                                                     │
│  OPENAI → APIGenerator                             │
│             gpt-5-mini                              │
│             max_completion_tokens                   │
└─────────────────────────────────────────────────────┘
                │
                ▼
┌─────────────────────────────────────────────────────┐
│  히스토리 업데이트                                    │
│    chat_history.append(user / assistant)            │
│    최대 20개 메시지 유지 (_MAX_HISTORY_MSGS = 20)    │
└─────────────────────────────────────────────────────┘
                │
                ▼
          최종 답변 반환
    {'answer', 'sources', 'sub_queries'}
```

---

## Query Router 분류 기준

| 조건 | 타입 | 사용 Retriever |
|---|---|---|
| 지시어 포함 (`그 `, `앞서 `, `해당 사업` 등) | C타입 | kh_v3 + use_hybrid=True |
| 직전 발화와 형태소 오버랩 없음 | C타입 | kh_v3 + use_hybrid=True |
| 그 외 | A/B/D/E타입 | chunks_all + use_hybrid=False |


## 채택 근거 (MRR 기준 — Lost in the Middle 대응)

| 타입 | chunks_all MRR | kh_v3 hybrid MRR | chunks_all Hit@5 | kh_v3 hybrid Hit@5 | 채택 |
|---|---|---|---|---|---|
| A | 0.8329 | **0.8380** | **0.9302** | 0.9012 | chunks_all (Hit@5 우세) |
| B | 0.8359 | **0.8660** | **0.9579** | 0.9252 | chunks_all (Hit@5 우세) |
| C (히스토리) | 0.7415 | 0.7342 | 0.8361 | **0.8689** | kh_v3 hybrid (Hit@5 +3.3%p) |
| D | **0.8487** | 0.7090 | **0.8769** | 0.8154 | chunks_all |
| E | **0.8138** | 0.5923 | **0.8154** | 0.6769 | chunks_all |

> MRR이 높을수록 정답이 Top-1에 가까워 LLM의 Lost in the Middle 영향을 줄일 수 있다.  
> A/B타입은 kh_v3 hybrid MRR이 앞서지만 Hit@5에서 chunks_all이 +2~3%p 우세해 chunks_all 채택.  
> C타입은 두 지표 모두 kh_v3 hybrid 우세 (MRR 근소, Hit@5 +3.3%p) → kh_v3 hybrid 채택.  
> D/E타입은 chunks_all이 MRR, Hit@5 모두 압도적 우세.
